# Movie Recommendation System (Task 3) — MovieLens 100K (ml-latest-small)

Build a **Movie Recommendation System** on the MovieLens dataset that:
- Uses **user similarity** from a user–item matrix
- Recommends **top-rated unseen movies** for a given user
- Evaluates performance with **Precision@K**

**Bonus requirements:**
- Implement **item-based collaborative filtering**
- Try **matrix factorization (SVD)**

## Technologies Used (and Why)

- **Python**: main programming language
- **Pandas**: load and manipulate tables (CSV files)
- **NumPy**: numerical operations and matrix math
- **Scikit-learn**:
  - `cosine_similarity` → compute similarity between users/items
  - `TruncatedSVD` → matrix factorization for latent features

## Core Concepts (Simple Definitions)

- **User–Item Matrix**: rows = users, columns = movies, values = ratings
- **Collaborative Filtering**: recommend using patterns from other users/items
- **Cosine Similarity**: measures how similar two users/items are based on ratings
- **Precision@K**: out of top K recommendations, how many were actually relevant
- **SVD (Matrix Factorization)**: factorizes the matrix to find hidden patterns

## Step 1: Import Libraries

In [77]:
# Core libraries
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Step 2: Load Dataset

In [78]:
# Load MovieLens data
base_path = "/kaggle/input/movielens-100k/ml-latest-small"

movies = pd.read_csv(f"{base_path}/movies.csv")
ratings = pd.read_csv(f"{base_path}/ratings.csv")

print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)
movies.head()

Movies shape: (9742, 3)
Ratings shape: (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Step 3: Dataset overview

- **`movies.csv`**: `movieId`, `title`, `genres`
- **`ratings.csv`**: `userId`, `movieId`, `rating`, `timestamp`

In [95]:
import os
import pandas as pd

base_path = "/kaggle/input/movielens-100k/ml-latest-small"

# List files
print("Files:", os.listdir(base_path))

# Load files
movies = pd.read_csv(f"{base_path}/movies.csv")
ratings = pd.read_csv(f"{base_path}/ratings.csv")
tags = pd.read_csv(f"{base_path}/tags.csv")
links = pd.read_csv(f"{base_path}/links.csv")

# Basic shapes
print("\nShapes:")
print("movies:", movies.shape)
print("ratings:", ratings.shape)
print("tags:", tags.shape)
print("links:", links.shape)

# Columns
print("\nColumns:")
print("movies:", movies.columns.tolist())
print("ratings:", ratings.columns.tolist())
print("tags:", tags.columns.tolist())
print("links:", links.columns.tolist())

# Unique counts
print("\nUnique counts:")
print("users:", ratings["userId"].nunique())
print("movies rated:", ratings["movieId"].nunique())
print("total ratings:", len(ratings))
print("total tags:", len(tags))

# Missing values
print("\nMissing values:")
print("movies:\n", movies.isna().sum())
print("ratings:\n", ratings.isna().sum())
print("tags:\n", tags.isna().sum())
print("links:\n", links.isna().sum())

Files: ['movies.csv', 'ratings.csv', 'README.txt', 'tags.csv', 'links.csv']

Shapes:
movies: (9742, 3)
ratings: (100836, 4)
tags: (3683, 4)
links: (9742, 3)

Columns:
movies: ['movieId', 'title', 'genres']
ratings: ['userId', 'movieId', 'rating', 'timestamp']
tags: ['userId', 'movieId', 'tag', 'timestamp']
links: ['movieId', 'imdbId', 'tmdbId']

Unique counts:
users: 610
movies rated: 9724
total ratings: 100836
total tags: 3683

Missing values:
movies:
 movieId    0
title      0
genres     0
dtype: int64
ratings:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
tags:
 userId       0
movieId      0
tag          0
timestamp    0
dtype: int64
links:
 movieId    0
imdbId     0
tmdbId     8
dtype: int64


In [79]:
# Dataset summary numbers
print("Movies columns:", movies.columns.tolist())
print("Ratings columns:", ratings.columns.tolist())

print("\nUnique users:", ratings["userId"].nunique())
print("Unique movies rated:", ratings["movieId"].nunique())
print("Total ratings:", len(ratings))

# Merge for easy title lookup later
ratings_with_titles = ratings.merge(movies, on="movieId", how="left")
ratings_with_titles.head()

Movies columns: ['movieId', 'title', 'genres']
Ratings columns: ['userId', 'movieId', 'rating', 'timestamp']

Unique users: 610
Unique movies rated: 9724
Total ratings: 100836


,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


## Step 4: Train/Test Split (Per User)

In [80]:
# Split ratings into train/test per user

def train_test_split_by_user(ratings_df, test_ratio=0.2, min_test=1):
    train_parts = []
    test_parts = []

    for user_id, group in ratings_df.groupby("userId"):
        n_test = max(min_test, int(len(group) * test_ratio))
        test_idx = group.sample(n=n_test, random_state=RANDOM_STATE).index
        test_parts.append(group.loc[test_idx])
        train_parts.append(group.drop(test_idx))

    train_df = pd.concat(train_parts).reset_index(drop=True)
    test_df = pd.concat(test_parts).reset_index(drop=True)
    return train_df, test_df

train_ratings, test_ratings = train_test_split_by_user(ratings, test_ratio=0.2)

print("Train ratings:", train_ratings.shape)
print("Test ratings:", test_ratings.shape)

# Build user-item matrix (train only)
user_item_matrix = train_ratings.pivot_table(
    index="userId", columns="movieId", values="rating", fill_value=0.0
)

print("User-item matrix shape:", user_item_matrix.shape)

Train ratings: (80896, 4)
Test ratings: (19940, 4)
User-item matrix shape: (610, 8974)


## Step 5: User-Based Similarity (Cosine)

In [81]:
# Compute user-user cosine similarity
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index
)

user_similarity_df.iloc[:5, :5]

userId,1,2,3,4,5
userId,,,,,
1,1.000000,0.000000,0.077030,0.131753,0.098375
2,0.000000,1.000000,0.000000,0.000000,0.020972
3,0.077030,0.000000,1.000000,0.000000,0.006399
4,0.131753,0.000000,0.000000,1.000000,0.126249
5,0.098375,0.020972,0.006399,0.126249,1.000000


## Step 6: User-Based Recommendations (Unseen Movies)

In [82]:
# Recommend unseen movies for a given user using user-based CF

def recommend_movies_user_based(user_id, user_item, similarity_df, movies_df, top_k=10):
    if user_id not in user_item.index:
        return pd.DataFrame(columns=["movieId", "title", "score"])

    # Similarity scores for the target user
    sim_scores = similarity_df.loc[user_id].copy()
    sim_scores.loc[user_id] = 0.0  # remove self-similarity

    # Only keep positive similarities
    sim_scores = sim_scores[sim_scores > 0]
    if sim_scores.sum() == 0:
        return pd.DataFrame(columns=["movieId", "title", "score"])

    # Weighted sum of ratings from similar users
    weighted_ratings = sim_scores.values @ user_item.loc[sim_scores.index].values
    norm = sim_scores.sum()
    scores = weighted_ratings / norm

    # Convert to Series for easy filtering
    scores_series = pd.Series(scores, index=user_item.columns)

    # Exclude already rated movies
    rated_movies = user_item.loc[user_id]
    unseen_scores = scores_series[rated_movies == 0]

    # Top K recommendations
    top_recs = unseen_scores.sort_values(ascending=False).head(top_k)
    recs = top_recs.reset_index(name="score").rename(columns={"index": "movieId"})
    recs = recs.merge(movies_df, on="movieId", how="left")
    return recs[["movieId", "title", "score"]]

# Example recommendation for a user
example_user = user_item_matrix.index[0]
recommend_movies_user_based(example_user, user_item_matrix, user_similarity_df, movies, top_k=10)

,movieId,title,score
0,296,Pulp Fiction (1994),2.211241
1,318,"Shawshank Redemption, The (1994)",2.039536
2,260,Star Wars: Episode IV - A New Hope (1977),1.974946
3,589,Terminator 2: Judgment Day (1991),1.701038
4,858,"Godfather, The (1972)",1.539273
5,1210,Star Wars: Episode VI - Return of the Jedi (1983),1.507968
6,2762,"Sixth Sense, The (1999)",1.419706
7,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),1.357870
8,4993,"Lord of the Rings: The Fellowship of the Ring,...",1.345206
9,7153,"Lord of the Rings: The Return of the King, The...",1.326473


## Step 7: Evaluation with Precision@K

In [83]:
# Relevant items = test ratings with rating >= 4.0

def precision_at_k(user_id, k, train_df, test_df, user_item, similarity_df, movies_df):
    # Ground-truth relevant movies from test set
    relevant = test_df[(test_df["userId"] == user_id) & (test_df["rating"] >= 4.0)]["movieId"]
    relevant = set(relevant.tolist())
    if len(relevant) == 0:
        return None

    # Get recommendations
    recs = recommend_movies_user_based(user_id, user_item, similarity_df, movies_df, top_k=k)
    recommended = set(recs["movieId"].tolist())

    # Precision@K
    hits = len(recommended & relevant)
    return hits / k

# Compute mean Precision@K across users with at least 1 relevant test item
K = 10
precisions = []
for user_id in user_item_matrix.index:
    p = precision_at_k(user_id, K, train_ratings, test_ratings, user_item_matrix, user_similarity_df, movies)
    if p is not None:
        precisions.append(p)

print(f"Mean Precision@{K}: {np.mean(precisions):.4f}")
print("Users evaluated:", len(precisions))

Mean Precision@10: 0.1651
Users evaluated: 598


## Bonus 1: Item-Based Collaborative Filtering

In [84]:
# Build item-user matrix for item-based CF
item_user_matrix = user_item_matrix.T

# Item-item cosine similarity
item_similarity = cosine_similarity(item_user_matrix)
item_similarity_df = pd.DataFrame(
    item_similarity, index=item_user_matrix.index, columns=item_user_matrix.index
)


def recommend_movies_item_based(user_id, user_item, item_sim_df, movies_df, top_k=10):
    if user_id not in user_item.index:
        return pd.DataFrame(columns=["movieId", "title", "score"])

    user_ratings = user_item.loc[user_id]
    rated_items = user_ratings[user_ratings > 0]
    if rated_items.empty:
        return pd.DataFrame(columns=["movieId", "title", "score"])

    # Weighted sum of item similarities
    scores = item_sim_df.loc[rated_items.index].T.dot(rated_items)
    scores = scores / rated_items.sum()

    # Exclude already rated movies
    scores = scores[user_ratings == 0]

    # Top K recommendations
    top_recs = scores.sort_values(ascending=False).head(top_k)
    recs = top_recs.reset_index(name="score").rename(columns={"index": "movieId"})
    recs = recs.merge(movies_df, on="movieId", how="left")
    return recs[["movieId", "title", "score"]]

# Example item-based recommendation
recommend_movies_item_based(example_user, user_item_matrix, item_similarity_df, movies, top_k=10)

,movieId,title,score
0,1036,Die Hard (1988),0.273107
1,1968,"Breakfast Club, The (1985)",0.270116
2,2762,"Sixth Sense, The (1999)",0.269485
3,260,Star Wars: Episode IV - A New Hope (1977),0.269438
4,1387,Jaws (1975),0.263640
5,2918,Ferris Bueller's Day Off (1986),0.263568
6,1527,"Fifth Element, The (1997)",0.262930
7,2470,Crocodile Dundee (1986),0.262712
8,1265,Groundhog Day (1993),0.259432
9,1580,Men in Black (a.k.a. MIB) (1997),0.258890


## Bonus 2: Matrix Factorization (SVD)

In [85]:
# SVD-based recommendation
n_components = 20  # latent factors
svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)

user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_

# Reconstruct predicted ratings matrix
predicted_ratings = np.dot(user_factors, item_factors)
predicted_ratings_df = pd.DataFrame(
    predicted_ratings, index=user_item_matrix.index, columns=user_item_matrix.columns
)


def recommend_movies_svd(user_id, user_item, pred_df, movies_df, top_k=10):
    if user_id not in pred_df.index:
        return pd.DataFrame(columns=["movieId", "title", "score"])

    # Exclude already rated movies
    rated = user_item.loc[user_id]
    preds = pred_df.loc[user_id]
    preds = preds[rated == 0]

    # Top K recommendations
    top_recs = preds.sort_values(ascending=False).head(top_k)
    recs = top_recs.reset_index(name="score").rename(columns={"index": "movieId"})
    recs = recs.merge(movies_df, on="movieId", how="left")
    return recs[["movieId", "title", "score"]]

# Example SVD recommendation
recommend_movies_svd(example_user, user_item_matrix, predicted_ratings_df, movies, top_k=10)

,movieId,title,score
0,589,Terminator 2: Judgment Day (1991),3.776744
1,260,Star Wars: Episode IV - A New Hope (1977),3.598326
2,1210,Star Wars: Episode VI - Return of the Jedi (1983),3.142913
3,858,"Godfather, The (1972)",2.850434
4,1036,Die Hard (1988),2.848144
5,1265,Groundhog Day (1993),2.777977
6,2762,"Sixth Sense, The (1999)",2.576929
7,1200,Aliens (1986),2.564695
8,1387,Jaws (1975),2.511814
9,1968,"Breakfast Club, The (1985)",2.482262


## Step 8: How to Test with Your Own UserId

Use any `userId` present in `ratings.csv` and replace it below.

```python
my_user = 5
recommend_movies_user_based(my_user, user_item_matrix, user_similarity_df, movies, top_k=10)
precision_at_k(my_user, 10, train_ratings, test_ratings, user_item_matrix, user_similarity_df, movies)
```


## Conclusion

We built a complete movie recommendation pipeline using MovieLens ratings. We created a user–item matrix, computed user similarity, recommended unseen movies, and evaluated performance with Precision@K. As bonus extensions, we implemented item-based collaborative filtering and a matrix-factorization (SVD) model to capture latent user–movie preferences.